In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LogNorm, LinearSegmentedColormap
import os

configs = [
    {
        'process_data_dir': r'PATH_TO_PROCESS_DATA_DIR_1',
        'output_dir': r'PATH_TO_OUTPUT_DIR_1',
        'out_name': 'gantt_heatmap_source1',
        'global_max_clusters': None,
    },
    {
        'process_data_dir': r'PATH_TO_PROCESS_DATA_DIR_2',
        'output_dir': r'PATH_TO_OUTPUT_DIR_2',
        'out_name': 'gantt_heatmap_source2',
        'global_max_clusters': 51,
    },
]

shared_legend_dir = r'PATH_TO_SHARED_LEGEND_DIR'
os.makedirs(shared_legend_dir, exist_ok=True)

time_scale_ranges = ['0-1', '2-3', '4-6', '7-12', '13-18', '19-24', '25-48']
seasons = ['Spring', 'Summer', 'Autumn', 'Winter']
season_labels = {'Spring': 'MAM', 'Summer': 'JJA', 'Autumn': 'SON', 'Winter': 'DJF'}
n_windows = 11
TARGET_W_IN = 8.0
SAVE_DPI = 600

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['mathtext.fontset'] = 'stixsans'
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

bg = "#FFFFFF"
nodata_bg = '#EEF3F8'
spine_col = '#000000'
txt_col = '#000000'

colors_list = [
    '#DCEAF6', '#C4DAEE', '#A7C7E3', '#84AFD4',
    '#5F93C2', '#3E78AE', '#255E97', '#13467C', '#083061'
]
heat_cmap = LinearSegmentedColormap.from_list('NatureCoolBlues_Bold', colors_list, N=256)
heat_cmap.set_bad(color=nodata_bg)

vmin = 100.0
vmax = 1500.0
norm = LogNorm(vmin=vmin, vmax=vmax)

def draw_heatmap(process_data_dir, output_dir, out_name, global_max_clusters_override=None):
    os.makedirs(output_dir, exist_ok=True)

    if global_max_clusters_override is not None:
        global_max_clusters = global_max_clusters_override
    else:
        global_max_clusters = 1
        for ts in time_scale_ranges:
            for s in seasons:
                p = os.path.join(process_data_dir, f'{ts}_{s}', f'heatmap_matrix_{ts}_{s}.csv')
                if os.path.exists(p):
                    df = pd.read_csv(p, index_col=0, encoding='utf-8-sig')
                    global_max_clusters = max(global_max_clusters, len(df))

    n_rows, n_cols = len(seasons), len(time_scale_ranges)
    cell_w, cell_h = 0.10, 0.06
    subplot_w = n_windows * cell_w
    subplot_h = global_max_clusters * cell_h
    gap_w, gap_h = 0.15, 0.20
    margin_l, margin_r = 0.80, 0.60
    margin_t, margin_b = 0.45, 0.65
    cb_gap, cb_w_in = 0.10, 0.12

    natural_w = (
        margin_l + n_cols * subplot_w + (n_cols - 1) * gap_w
        + margin_r + cb_gap + cb_w_in
    )
    natural_h = margin_t + n_rows * subplot_h + (n_rows - 1) * gap_h + margin_b

    scale = TARGET_W_IN / natural_w

    def S(x):
        return x * scale

    sw_s = n_windows * S(cell_w)
    sh_s = global_max_clusters * S(cell_h)
    gw_s, gh_s = S(gap_w), S(gap_h)
    ml_s, mb_s = S(margin_l), S(margin_b)

    fig_w = TARGET_W_IN
    fig_h = natural_h * scale

    def to_norm_x(ci):
        return (ml_s + ci * (sw_s + gw_s)) / fig_w, sw_s / fig_w

    def to_norm_y(ri):
        return (mb_s + (n_rows - 1 - ri) * (sh_s + gh_s)) / fig_h, sh_s / fig_h

    all_cols = [f'{y}-{y+9}' for y in range(2001, 2012)]
    full_lbls = [f'{y}-{y+9}' for y in range(2001, 2012)]
    sparse_lbls = [full_lbls[i] if i % 2 == 0 else '' for i in range(n_windows)]
    x_edges = np.arange(0, n_windows + 1)
    y_edges = np.arange(0, global_max_clusters + 1)
    SPINE_W = 1.0

    fig = plt.figure(figsize=(fig_w, fig_h), facecolor=bg)

    for row_i, season in enumerate(seasons):
        for col_i, ts in enumerate(time_scale_ranges):
            xl, aw = to_norm_x(col_i)
            yb, ah = to_norm_y(row_i)
            ax = fig.add_axes([xl, yb, aw, ah])
            ax.set_facecolor(nodata_bg)

            for sp in ax.spines.values():
                sp.set_color(spine_col)
                sp.set_linewidth(SPINE_W)
                sp.set_visible(True)

            hm_path = os.path.join(process_data_dir, f'{ts}_{season}', f'heatmap_matrix_{ts}_{season}.csv')
            ax.set_xlim(0, n_windows)
            ax.set_ylim(global_max_clusters, 0)

            if row_i == n_rows - 1:
                ax.set_xticks(np.arange(n_windows) + 0.5)
                ax.set_xticklabels(
                    sparse_lbls,
                    fontsize=10,
                    rotation=90,
                    ha='center',
                    va='top',
                    color=txt_col
                )
                ax.tick_params(
                    axis='x',
                    length=3,
                    width=0.8,
                    pad=2,
                    colors=txt_col,
                    bottom=True,
                    top=False,
                    labelbottom=True
                )
            else:
                ax.set_xticks([])
                ax.tick_params(axis='x', bottom=True, top=False, labelbottom=False)

            ax.set_yticks([])
            ax.tick_params(axis='y', left=True, right=False, labelleft=False)

            if row_i == 0:
                ax.set_title(f'{ts}', fontsize=15, pad=10, color=txt_col)
            if col_i == 0:
                ax.set_ylabel(season_labels[season], fontsize=20, labelpad=4, color=txt_col)

            if not os.path.exists(hm_path):
                ax.text(
                    0.5,
                    0.5,
                    'No data',
                    ha='center',
                    va='center',
                    transform=ax.transAxes,
                    fontsize=15,
                    color='#CCCCCC',
                    style='italic'
                )
                continue

            df_hm = pd.read_csv(hm_path, index_col=0, encoding='utf-8-sig')
            n_cl = len(df_hm)

            if n_cl < global_max_clusters:
                pad_df = pd.DataFrame(
                    np.zeros((global_max_clusters - n_cl, n_windows)),
                    columns=df_hm.columns,
                    index=[f'PAD_{i}' for i in range(global_max_clusters - n_cl)]
                )
                df_hm = pd.concat([df_hm, pad_df])

            for c in all_cols:
                if c not in df_hm.columns:
                    df_hm[c] = 0
            df_hm = df_hm[all_cols]

            mat = df_hm.values.astype(float)
            ax.pcolormesh(
                x_edges,
                y_edges,
                np.ma.masked_where(mat == 0, mat),
                cmap=heat_cmap,
                norm=norm,
                edgecolors='none',
                linewidth=0,
                antialiased=False,
                shading='flat'
            )

            ax.text(
                0.15,
                0.05,
                f'n={n_cl}',
                transform=ax.transAxes,
                fontsize=15,
                color=txt_col,
                ha='left',
                va='bottom',
                bbox=dict(
                    boxstyle='round,pad=0.3',
                    facecolor='white',
                    edgecolor='#CCCCCC',
                    linewidth=0.5,
                    alpha=0.9
                )
            )

    for ext in ('png', 'pdf'):
        fig.savefig(
            os.path.join(output_dir, f'{out_name}.{ext}'),
            dpi=SAVE_DPI,
            facecolor=bg,
            bbox_inches='tight'
        )
    plt.show()

draw_heatmap(
    configs[0]['process_data_dir'],
    configs[0]['output_dir'],
    configs[0]['out_name'],
    global_max_clusters_override=configs[0]['global_max_clusters']
)

draw_heatmap(
    configs[1]['process_data_dir'],
    configs[1]['output_dir'],
    configs[1]['out_name'],
    global_max_clusters_override=configs[1]['global_max_clusters']
)

manual_ticks = [100, 200, 400, 700, 1000, 1500]

fig_cb, ax_cb = plt.subplots(figsize=(3.5, 0.55), facecolor=bg)
fig_cb.subplots_adjust(left=0.05, right=0.95, bottom=0.55, top=0.85)

sm = cm.ScalarMappable(cmap=heat_cmap, norm=norm)
sm.set_array([])

cbar = fig_cb.colorbar(sm, cax=ax_cb, orientation='horizontal')
cbar.set_ticks(manual_ticks)
cbar.set_ticklabels([str(v) for v in manual_ticks])
cbar.set_label('Pixel count', fontsize=8, color=txt_col, labelpad=4)
cbar.ax.tick_params(
    axis='x',
    which='major',
    labelsize=7,
    width=0.8,
    length=3.5,
    colors=txt_col,
    direction='out'
)
cbar.outline.set_edgecolor(spine_col)
cbar.outline.set_linewidth(0.8)

for ext in ('png', 'pdf'):
    fig_cb.savefig(
        os.path.join(shared_legend_dir, f'shared_colorbar.{ext}'),
        dpi=SAVE_DPI,
        facecolor=bg,
        bbox_inches='tight'
    )
plt.show()
